# CPA LINCS THP1 imputation

Plotnine diagnostics for CPA validation and ISOMIGA protective-count rankings.

In [ ]:
from pathlib import Path

import pandas as pd
from plotnine import *

DATA = Path('../data/processed/cpa')
FIG = Path('../results/figures')
FIG.mkdir(parents=True, exist_ok=True)

## CPA validation metrics

In [ ]:
metrics_path = DATA / 'cpa_validation_pilot_metrics.tsv'
if metrics_path.exists():
    metrics = pd.read_csv(metrics_path, sep='\t')
    display(metrics.head(10))
else:
    metrics = None
    print('Run scripts/slurm/cpa_validation_pilot.sbatch first.')

In [ ]:
if metrics is not None:
    plot_df = metrics[metrics['pert_id'] != 'OVERALL'].dropna(subset=['pearson_r']).sort_values('pearson_r', ascending=False).head(30)
    plot_df['pert_iname'] = pd.Categorical(plot_df['pert_iname'], categories=plot_df['pert_iname'][::-1], ordered=True)
    p = (
        ggplot(plot_df, aes('pearson_r', 'pert_iname'))
        + geom_col(fill='#4c78a8')
        + labs(x='Pearson r, CPA-predicted vs observed THP1 target genes', y='Drug')
        + theme_bw()
        + theme(figure_size=(6, 6))
    )
    display(p)
    p.save(FIG / 'cpa_validation_pilot_top_correlations.png', dpi=300)

## ISOMIGA protective-count ranking

In [ ]:
rank_path = DATA / 'isomiga_cpa_protective_count_combined_drug_scores.tsv'
gene_path = DATA / 'isomiga_cpa_protective_count_combined_drug_gene_scores.tsv'
if rank_path.exists() and gene_path.exists():
    ranks = pd.read_csv(rank_path, sep='\t')
    genes = pd.read_csv(gene_path, sep='\t')
    display(ranks.head(20))
else:
    ranks = genes = None
    print('Run scripts/rank_cpa_drugs_by_isomiga.py after CPA prediction.')

In [ ]:
if ranks is not None:
    top = ranks.head(30).copy()
    top['label'] = top['pert_iname'] + ' (' + top['response_source'] + ')'
    top['label'] = pd.Categorical(top['label'], categories=top['label'][::-1], ordered=True)
    p = (
        ggplot(top, aes('n_protective_genes', 'label', fill='response_source'))
        + geom_col()
        + labs(x='Number of ISOMIGA genes moved in protective direction', y='Drug')
        + theme_bw()
        + theme(figure_size=(7, 7))
    )
    display(p)
    p.save(FIG / 'cpa_isomiga_top_protective_count_drugs.png', dpi=300)

In [ ]:
if ranks is not None and genes is not None:
    selected = ranks.head(5)[['pert_id', 'pert_iname', 'moa']]
    plot_df = genes.merge(selected, on=['pert_id', 'pert_iname'], how='inner', suffixes=('', '_rank'))
    plot_df['drug_label'] = plot_df['pert_iname'] + ' (' + plot_df['moa'].fillna('not annotated') + ')'
    order = selected['pert_iname'].tolist()
    label_order = [plot_df.loc[plot_df['pert_iname'].eq(x), 'drug_label'].iloc[0] for x in order if plot_df['pert_iname'].eq(x).any()]
    plot_df['drug_label'] = pd.Categorical(plot_df['drug_label'], categories=label_order, ordered=True)
    plot_df['gene_name'] = pd.Categorical(plot_df['gene_name'], categories=sorted(plot_df['gene_name'].unique())[::-1], ordered=True)
    p = (
        ggplot(plot_df, aes('gene_name', 'protective_push_z', fill='pushes_protective'))
        + geom_col()
        + geom_hline(yintercept=0, color='#333333', size=0.3)
        + facet_wrap('~drug_label', nrow=1)
        + scale_fill_manual(values={True: '#2f7d4f', False: '#9c2f2f'})
        + labs(x='ISOMIGA gene', y='LINCS/CPA z in protective direction', fill='Protective')
        + theme_bw()
        + theme(figure_size=(8, 2.8), axis_text_x=element_text(rotation=45, ha='right'), strip_text=element_text(size=7))
    )
    display(p)
    p.save(FIG / 'cpa_selected_top_drug_gene_match_bars.pdf')